# 🤖 XAS AI Agent

Talk to this agent in natural language to visualize and analyze your XAS data.

**Example commands:**
- *"List all scans"*
- *"Show me the TEY for scan 45612"*
- *"Plot MCP of 45645 normalized by I0"*
- *"Compare TEY of scans 45611, 45612, 45613"*
- *"Save the last plot as my_data.txt"*
- *"Tell me about scan 45645"*

In [12]:
# ── Setup (no file loading — instant startup) ─────────────────────────────────
import os, sys, json, textwrap, importlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, os.path.abspath("."))
import xas_utils as xu
importlib.reload(xu)  # always pick up latest version

matplotlib.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
%matplotlib inline

# ── Load API key from .env ────────────────────────────────────────────────────
load_dotenv()
CBORG_API_KEY = os.environ.get("CBORG_API_KEY", "")
if not CBORG_API_KEY or CBORG_API_KEY == "your-api-key-here":
    raise ValueError("Please set CBORG_API_KEY in your .env file.")

client = OpenAI(
    api_key=CBORG_API_KEY,
    base_url="https://api.cborg.lbl.gov/v1",
)
MODEL = "claude-sonnet"
DATA_DIR = "731_Data"

print(f"✅ Agent ready (model: {MODEL}, data dir: {DATA_DIR})")
print("   No files loaded yet — data is loaded on demand when you ask.")

✅ Agent ready (model: claude-sonnet, data dir: 731_Data)
   No files loaded yet — data is loaded on demand when you ask.


In [13]:
# ── Tools & Agent ─────────────────────────────────────────────────────────────

_last_plot = {}  # tracks last plotted data for saving
_cache = {}      # scan_id -> {"meta": dict, "df": DataFrame}

def _load(scan_id: str) -> tuple:
    """Load a scan on demand, with caching."""
    sid = xu.resolve_scan_id(scan_id)
    if sid not in _cache:
        fp = xu.scan_filepath(sid, DATA_DIR)
        _cache[sid] = {"meta": xu.parse_header(fp), "df": xu.load_scan(fp)}
    return sid, _cache[sid]["meta"], _cache[sid]["df"]


# ── Tool definitions ──────────────────────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_scans",
            "description": "List all available XAS scan file names in the data directory. Only lists filenames — does not load any data.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "plot_scan",
            "description": "Plot a single XAS scan. Plots the specified signal (TEY, TFY, or MCP) vs Mono Energy in eV. Optionally normalize by I0.",
            "parameters": {
                "type": "object",
                "properties": {
                    "scan_id": {"type": "string", "description": "Scan identifier, e.g. 'SigScan45611' or '45611'."},
                    "signal": {"type": "string", "enum": ["TEY", "TFY", "MCP"], "description": "Signal channel to plot."},
                    "normalize": {"type": "boolean", "description": "If true, divide signal by I0. Default false."}
                },
                "required": ["scan_id", "signal"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_scans",
            "description": "Overlay multiple XAS scans on one plot for comparison.",
            "parameters": {
                "type": "object",
                "properties": {
                    "scan_ids": {"type": "array", "items": {"type": "string"}, "description": "List of scan identifiers."},
                    "signal": {"type": "string", "enum": ["TEY", "TFY", "MCP"], "description": "Signal channel."},
                    "normalize": {"type": "boolean", "description": "If true, divide by I0. Default false."}
                },
                "required": ["scan_ids", "signal"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_data",
            "description": "Save the last plotted data to an ASCII text file in exported_data/. Auto-generates filename with timestamp if none given.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "Optional filename (e.g. 'my_data.txt')."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "show_scan_info",
            "description": "Show detailed metadata, energy range, and available signals for a specific scan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "scan_id": {"type": "string", "description": "Scan identifier."}
                },
                "required": ["scan_id"]
            }
        }
    }
]


# ── Tool implementations ──────────────────────────────────────────────────────

def tool_list_scans(**kw) -> str:
    ids = xu.list_scan_files(DATA_DIR)
    return f"Found {len(ids)} scans:\n" + "\n".join(ids)


def tool_plot_scan(scan_id: str, signal: str, normalize: bool = False, **kw) -> str:
    global _last_plot
    try:
        sid, meta, df = _load(scan_id)
    except FileNotFoundError as e:
        return str(e)

    energy = xu.get_energy(df)
    if normalize:
        signal_data = xu.normalize_by_i0(df, signal)
        ylabel = f"{signal} / I0"
    else:
        signal_data = xu.get_signal(df, signal)
        ylabel = signal

    fig, ax = plt.subplots()
    ax.plot(energy, signal_data, "b-", linewidth=1.2)
    ax.set_xlabel("Mono Energy (eV)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{sid}  —  {meta['scan_type']}  ({meta['date']})")
    plt.tight_layout()
    plt.show()

    _last_plot = {"energy": energy, "signal": signal_data, "signal_name": ylabel, "scan_id": sid}
    return f"Plotted {ylabel} for {sid}. Energy: {energy.min():.2f}–{energy.max():.2f} eV, {len(energy)} pts."


def tool_compare_scans(scan_ids: list, signal: str, normalize: bool = False, **kw) -> str:
    global _last_plot
    loaded = []
    for raw in scan_ids:
        try:
            loaded.append(_load(raw))
        except FileNotFoundError as e:
            return str(e)

    fig, ax = plt.subplots()
    for sid, meta, df in loaded:
        energy = xu.get_energy(df)
        if normalize:
            sig_data = xu.normalize_by_i0(df, signal)
            ylabel = f"{signal} / I0"
        else:
            sig_data = xu.get_signal(df, signal)
            ylabel = signal
        ax.plot(energy, sig_data, linewidth=1.0, label=sid)

    ax.set_xlabel("Mono Energy (eV)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Comparison — {ylabel}")
    ax.legend(fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()

    first_sid, _, df0 = loaded[0]
    _last_plot = {
        "energy": xu.get_energy(df0),
        "signal": xu.normalize_by_i0(df0, signal) if normalize else xu.get_signal(df0, signal),
        "signal_name": ylabel,
        "scan_id": "_".join(s for s, _, _ in loaded),
    }
    return f"Compared {len(loaded)} scans: {', '.join(s for s, _, _ in loaded)}."


def tool_save_data(filename: str = None, **kw) -> str:
    if not _last_plot:
        return "Error: Nothing to save. Please plot something first."
    path = xu.export_data(
        energy=_last_plot["energy"], signal=_last_plot["signal"],
        signal_name=_last_plot["signal_name"], scan_id=_last_plot["scan_id"],
        filename=filename,
    )
    return f"Data saved to: {path}"


def tool_show_scan_info(scan_id: str, **kw) -> str:
    try:
        sid, meta, df = _load(scan_id)
    except FileNotFoundError as e:
        return str(e)
    energy = xu.get_energy(df)
    signals = xu.list_available_signals(df)
    return "\n".join([
        f"Scan: {sid}",
        f"Date: {meta['date']}",
        f"Scan Type: {meta['scan_type']}",
        f"Scan File: {meta['scan_file']}",
        f"Count Time: {meta['count_time']}s",
        f"Delay After Move: {meta['delay_after_move']}s",
        f"Energy Range: {energy.min():.2f} – {energy.max():.2f} eV",
        f"Data Points: {len(df)}",
        f"Available Signals: {', '.join(signals)}",
    ])


TOOL_DISPATCH = {
    "list_scans": tool_list_scans,
    "plot_scan": tool_plot_scan,
    "compare_scans": tool_compare_scans,
    "save_data": tool_save_data,
    "show_scan_info": tool_show_scan_info,
}


# ── Agent ──────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = textwrap.dedent(f"""\
You are an XAS (X-ray Absorption Spectroscopy) data analysis assistant.
You help the user visualize and analyze XAS scan data collected at a synchrotron beamline.

Data directory: {DATA_DIR}
Scan files are named SigScan<number>.txt (e.g. SigScan45611.txt).
Available signal types: TEY (Total Electron Yield), TFY (Total Fluorescence Yield), MCP (MicroChannel Plate, column "MCP Np")
Energy column: Mono Energy in eV

Rules:
- The user may refer to scans by full ID (SigScan45611) or just the number (45611)
- When the user says "plot" or "show", use plot_scan or compare_scans
- When the user says "save" or "export", use save_data. Ask the user for a filename if they haven't provided one.
- When the user asks to "list" scans, use list_scans
- When the user asks about a specific scan's details, use show_scan_info
- I0 is the incident beam intensity; normalizing by I0 removes beam current variations
- Be helpful and concise
- If the request is ambiguous, make a reasonable assumption and explain what you did
""")

conversation = [{"role": "system", "content": SYSTEM_PROMPT}]


def chat(user_message: str):
    """Send a natural-language message to the agent."""
    conversation.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL, messages=conversation, tools=TOOLS, tool_choice="auto",
    )
    msg = response.choices[0].message

    while msg.tool_calls:
        conversation.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"🔧 {fn_name}({fn_args})")
            fn = TOOL_DISPATCH.get(fn_name)
            result = fn(**fn_args) if fn else f"Error: Unknown tool '{fn_name}'"
            conversation.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        response = client.chat.completions.create(
            model=MODEL, messages=conversation, tools=TOOLS, tool_choice="auto",
        )
        msg = response.choices[0].message

    conversation.append(msg)
    if msg.content:
        display(Markdown(msg.content))


print("✅ Tools & agent ready.")

✅ Tools & agent ready.


---
## 💬 Talk to the Agent

Run the cell below to launch the **interactive chat app** in your browser.

A new tab will open at [http://localhost:5050](http://localhost:5050) with a full chat interface.

Press **Interrupt** (⬛) in the notebook toolbar to stop the server when done.

In [ ]:
# ── Launch the Chat Web App ───────────────────────────────────────────────────
import subprocess, webbrowser, time, socket

PORT = 5050

# Start the Flask server as a subprocess
proc = subprocess.Popen(
    ['python3', 'chat_app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

print('🚀 Starting XAS Agent Chat server...')

# Wait until the server is actually listening
for _ in range(30):  # up to 15 seconds
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except (ConnectionRefusedError, OSError):
        time.sleep(0.5)
else:
    print('❌ Server did not start in time. Check chat_app.py for errors.')

print(f'✅ Server running at http://localhost:{PORT}')
print('   Press the ⬛ Stop button to shut down.')
webbrowser.open(f'http://localhost:{PORT}')

# Stream server output until interrupted
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print('\n🛑 Shutting down...')
finally:
    proc.terminate()
    proc.wait()